In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
from scipy.special import softmax

In [6]:
def add_correctness_probabilities(
    master_df: pd.DataFrame,
    judge_names: list[str]
) -> pd.DataFrame:
    """
    Add a correctness_probabilities column.

    Only rows where the original label is 'correct' are processed.
    Each row stores a JSON dictionary mapping judge names to P(yes).

    Softmax is applied to [logit_yes, logit_no] for each judge.
    """

    df = master_df.copy()
    df["correctness_probabilities"] = None

    correct_mask = (
        df["label"].astype(str).str.strip().str.lower() == "correct"
    )

    for idx in df.index[correct_mask]:
        probabilities = {}

        for judge_name in judge_names:
            yes_col = f"{judge_name}_logit_yes"
            no_col = f"{judge_name}_logit_no"

            if yes_col not in df.columns or no_col not in df.columns:
                continue

            yes = pd.to_numeric(
                pd.Series([df.at[idx, yes_col]]), errors="coerce"
            ).iloc[0]

            no = pd.to_numeric(
                pd.Series([df.at[idx, no_col]]), errors="coerce"
            ).iloc[0]

            if pd.isna(yes) or pd.isna(no):
                continue

            # Explicit two-class Softmax: [yes, no]
            logits = np.array([yes, no], dtype=np.float64)
            class_probabilities = softmax(logits)

            # Index 0 corresponds to yes; index 1 corresponds to no
            p_correct = float(class_probabilities[0])

            probabilities[judge_name] = p_correct

        if probabilities:
            df.at[idx, "correctness_probabilities"] = json.dumps(
                probabilities
            )

    return df


def save_correctness_probabilities(
    input_path="dataset/master.csv",
    output_path="dataset/master_with_correctness_probabilities.csv",
    judge_names=None
):
    if not judge_names:
        raise ValueError("Please provide the list of judge names.")

    master_df = pd.read_csv(input_path, dtype={"i": str})

    result_df = add_correctness_probabilities(
        master_df=master_df,
        judge_names=judge_names
    )

    result_df.to_csv(output_path, index=False)

    correct_mask = (
        result_df["label"].astype(str).str.strip().str.lower()
        == "correct"
    )

    print(f"Saved updated dataset to: {Path(output_path).resolve()}")
    print(f"Correct-label rows: {int(correct_mask.sum())}")
    print(
        "Correct-label rows with probabilities:",
        int(
            result_df.loc[
                correct_mask, "correctness_probabilities"
            ].notna().sum()
        )
    )

    return result_df

In [7]:
judge_names = [
    "Athene-70B",
    "dolphin-2.1-mistral-7b",
    "Hermes-3-Llama-3.1-70B",
    "Meta-Llama-3-70B-Instruct",
    "Mistral-7B-Instruct-v0.1",
    "Mixtral-8x7B-Instruct-v0.1",
    "OpenHermes-2-Mistral-7B",
    "Qwen2-72B-Instruct",
    "StableBeluga-7B",
    "Starling-LM-7B-alpha",
    "zephyr-7b-beta",
]

master_df = save_correctness_probabilities(
    input_path="dataset/master.csv",
    output_path="dataset/master_with_correctness_probabilities.csv",
    judge_names=judge_names
)

Saved updated dataset to: D:\AI Engineering\GAVEL\src\fine-tuning-laya\dataset\master_with_correctness_probabilities.csv
Correct-label rows: 14042
Correct-label rows with probabilities: 14042


In [ ]:
def derive_correctness_categories(
    master_df: pd.DataFrame,
    threshold: float = 0.80
) -> pd.DataFrame:
    """
    Derive categories only for rows where label == 'correct'.

    Rules:
      aggregate_score >= threshold -> fully_correct
      aggregate_score < threshold  -> partially_correct

    Adds:
      aggregate_correctness_probability:
          Arithmetic mean of valid judge probabilities.

      derived_correctness:
          Category derived from the aggregate probability.

    Rows labelled 'incorrect' are left untouched.
    The original label column is never modified.
    """

    df = master_df.copy()

    # Initialize new columns; original label remains unchanged.
    df["aggregate_correctness_probability"] = pd.NA
    df["derived_correctness"] = pd.NA

    correct_mask = (
        df["label"].astype(str).str.strip().str.lower() == "correct"
    )

    for idx in df.index[correct_mask]:
        raw_probabilities = df.at[idx, "correctness_probabilities"]

        if pd.isna(raw_probabilities) or not str(raw_probabilities).strip():
            continue

        # Parse the JSON dictionary of judge probabilities.
        try:
            probabilities = json.loads(raw_probabilities)
        except (json.JSONDecodeError, TypeError):
            continue

        if not isinstance(probabilities, dict):
            continue

        # Keep only valid numeric probabilities in [0, 1].
        valid_probabilities = []

        for value in probabilities.values():
            try:
                probability = float(value)
            except (TypeError, ValueError):
                continue

            if (
                pd.notna(probability)
                and 0.0 <= probability <= 1.0
            ):
                valid_probabilities.append(probability)

        if not valid_probabilities:
            continue

        # Arithmetic mean across available judges.
        aggregate_score = (
            sum(valid_probabilities) / len(valid_probabilities)
        )

        # Store the aggregate probability for reference.
        df.at[idx, "aggregate_correctness_probability"] = aggregate_score

        # Derive the category using the configured threshold.
        if aggregate_score >= threshold:
            df.at[idx, "derived_correctness"] = "fully_correct"
        else:
            df.at[idx, "derived_correctness"] = "partially_correct"

    return df


In [12]:
master_df = pd.read_csv(
    "dataset/master_with_correctness_probabilities.csv",
    dtype={"i": str}
)

result_df = derive_correctness_categories(
    master_df,
    threshold=0.80
)

result_df.to_csv(
    "dataset/master_with_derived_correctness.csv",
    index=False
)

# Inspect derived categories for originally correct rows.
correct_mask = (
    result_df["label"].astype(str).str.strip().str.lower()
    == "correct"
)

print(
    result_df.loc[correct_mask, [
        "correctness_probabilities",
        "aggregate_correctness_probability",
        "derived_correctness"
    ]].head(10)
)

# Verify incorrect rows were not assigned new values.
incorrect_mask = ~correct_mask

assert result_df.loc[
    incorrect_mask, "aggregate_correctness_probability"
].isna().all()

assert result_df.loc[
    incorrect_mask, "derived_correctness"
].isna().all()

print("Incorrect rows remain untouched.")

                            correctness_probabilities  \
0   {"Athene-70B": 0.9706877692486436, "dolphin-2....   
2   {"Athene-70B": 0.9241418199787566, "dolphin-2....   
4   {"Athene-70B": 0.1480471980316895, "dolphin-2....   
6   {"Athene-70B": 0.9859363729567544, "dolphin-2....   
8   {"Athene-70B": 0.679178699175393, "dolphin-2.1...   
10  {"Athene-70B": 4.8325638187255363e-05, "dolphi...   
12  {"Athene-70B": 0.9996646498695336, "dolphin-2....   
14  {"Athene-70B": 0.22270013882530884, "dolphin-2...   
16  {"Athene-70B": 0.9999741325689185, "dolphin-2....   
18  {"Athene-70B": 0.18242552380635632, "dolphin-2...   

   aggregate_correctness_probability derived_correctness  
0                           0.826613       fully_correct  
2                           0.515629   partially_correct  
4                           0.557088   partially_correct  
6                           0.388801   partially_correct  
8                           0.695365   partially_correct  
10                

In [33]:
final_columns = [
    "i",
    "subject",
    "question",
    "choices",
    "selected_choice",
    "answer",
    "label",
    "derived_correctness",
    "aggregate_correctness_probability",
]

final_df = result_df[final_columns].copy()

final_df.to_csv(
    "dataset/laya_finetuning_dataset.csv",
    index=False
)

In [34]:
final_df.head()

,i,subject,question,choices,selected_choice,answer,label,derived_correctness,aggregate_correctness_probability
0,0-correct,abstract_algebra,Find the degree for the given field extension ...,['0' '4' '2' '6'],1,4,correct,fully_correct,0.826613
1,0-incorrect,abstract_algebra,Find the degree for the given field extension ...,['0' '4' '2' '6'],3,6,incorrect,<NA>,<NA>
2,1-correct,abstract_algebra,"Let p = (1, 2, 5, 4)(2, 3) in S_5 . Find the i...",['8' '2' '24' '120'],2,24,correct,partially_correct,0.515629
3,1-incorrect,abstract_algebra,"Let p = (1, 2, 5, 4)(2, 3) in S_5 . Find the i...",['8' '2' '24' '120'],1,2,incorrect,<NA>,<NA>
4,2-correct,abstract_algebra,Find all zeros in the indicated finite field o...,"['0' '1' '0,1' '0,4']",3,"0,4",correct,partially_correct,0.557088


In [35]:
incorrect_mask = final_df["label"] == "incorrect"
final_df.loc[incorrect_mask, "derived_correctness"] = "incorrect"

In [36]:
final_df.head()

,i,subject,question,choices,selected_choice,answer,label,derived_correctness,aggregate_correctness_probability
0,0-correct,abstract_algebra,Find the degree for the given field extension ...,['0' '4' '2' '6'],1,4,correct,fully_correct,0.826613
1,0-incorrect,abstract_algebra,Find the degree for the given field extension ...,['0' '4' '2' '6'],3,6,incorrect,incorrect,<NA>
2,1-correct,abstract_algebra,"Let p = (1, 2, 5, 4)(2, 3) in S_5 . Find the i...",['8' '2' '24' '120'],2,24,correct,partially_correct,0.515629
3,1-incorrect,abstract_algebra,"Let p = (1, 2, 5, 4)(2, 3) in S_5 . Find the i...",['8' '2' '24' '120'],1,2,incorrect,incorrect,<NA>
4,2-correct,abstract_algebra,Find all zeros in the indicated finite field o...,"['0' '1' '0,1' '0,4']",3,"0,4",correct,partially_correct,0.557088


In [37]:
final_df.to_csv(
    "dataset/laya_finetuning_dataset.csv",
    index=False
)